<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/dataset_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Precomputation and Loading for Titans

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

Cloning into 'kauldron'...
remote: Enumerating objects: 9668, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 9668 (delta 54), reused 41 (delta 32), pack-reused 9474 (from 3)
Receiving objects: 100% (9668/9668), 2.92 MiB | 27.39 MiB/s, done.
Resolving deltas: 100% (6934/6934), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 85.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.6 MB/s eta

In [5]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog


Cloning into 'gemma'...
remote: Enumerating objects: 2564, done.
remote: Counting objects: 100% (1199/1199), done.
remote: Compressing objects: 100% (411/411), done.
remote: Total 2564 (delta 946), reused 819 (delta 788), pack-reused 1365 (from 2)
Receiving objects: 100% (2564/2564), 1.24 MiB | 21.45 MiB/s, done.
Resolving deltas: 100% (1800/1800), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.8 MB/s eta 0:00:00
Cloning into 'dialog'...
rem

In [ ]:
import os
os._exit(0)

In [1]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.bfloat16,
    # return_last_only=False,
    tokens="batch.tokens",
)

In [2]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')
MAX_LENGTH = 2048 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = 'validation'

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [3]:
def format_triviaqa_prompt(record):
    """Formats a TriviaQA record into a prompt for Gemma."""
    question = record.get('question', b'').decode('utf-8') if isinstance(record.get('question'), bytes) else record.get('question', '')

    # Extract context from search results if available
    contexts = record.get('search_results', {}).get('search_context', [])
    context_str = ""
    if contexts:
        ctx = contexts[0]
        context_str = ctx.decode('utf-8') if isinstance(ctx, bytes) else str(ctx)

    prompt = f"Context: {context_str}\n\nQuestion: {question}\n\nAnswer:"
    return prompt

In [4]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [6]:
split_name = 'test'
def precompute_and_save(split_name):
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей.")
for split_name in ['train', 'validation','test']:
    precompute_and_save(split_name) # Раскомментируйте для запуска

Начинаю фильтрацию и сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/train.array_record...


  0%|          | 83/138384 [00:00<02:47, 827.46it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 138384/138384 [02:18<00:00, 999.56it/s] 


Готово! Сохранено 49370 валидных записей.
Начинаю фильтрацию и сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/validation.array_record...


  0%|          | 0/18669 [00:00<?, ?it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 18669/18669 [00:51<00:00, 361.33it/s]

Готово! Сохранено 6686 валидных записей.


## 3. Загрузка сохраненных данных

Эта функция заменяет ваш `get_train_dataset_tydi_qa` в основном цикле обучения.

## 4. Использование в kd.train.Trainer

Пример того, как это интегрируется в основной конфиг.

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [5]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
output_path = os.path.join(str(DATA_DIR), output_file)
reader = array_record.ArrayRecordReader(str(input_path))
writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

In [6]:
reader.num_records()

6686

In [5]:
import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
import os
import jax

def generate_and_save_batched_dataset(
    model,
    params,
    tokenizer,
    split_name,
    batch_size=4,
    max_new_tokens=64,
    limit=None
):
    input_path = os.path.join(str(DATA_DIR), f"{split_name}.array_record")
    output_path = os.path.join(str(DATA_DIR), f"{split_name}_gemma_generated.array_record")

    print(f"Обработка {input_path}...")

    sampler = gm.text.Sampler(model=model, params=params, tokenizer=tokenizer)
    reader = array_record.ArrayRecordReader(str(input_path))
    num_records = reader.num_records() if limit is None else min(limit, reader.num_records())

    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    try:
        for i in tqdm.tqdm(range(0, num_records, batch_size)):
            # Очистка кэша JAX каждые 10 итераций для борьбы с фрагментацией памяти
            if (i // batch_size) % 10 == 0:
                jax.clear_caches()

            batch_records = []
            batch_prompts = []

            for j in range(i, min(i + batch_size, num_records)):
                record_bytes = reader.read()
                record = pickle.loads(record_bytes)
                batch_records.append(record)
                batch_prompts.append(format_triviaqa_prompt(record))

            if not batch_prompts: continue

            # Генерируем ответы
            responses = sampler.sample(batch_prompts, max_new_tokens=max_new_tokens)

            for record, response_text in zip(batch_records, responses):
                if 'answer' not in record: record['answer'] = {}
                record['answer']['value'] = response_text
                writer.write(pickle.dumps(record))
    finally:
        writer.close()
        reader.close()

    print(f"Готово! Файл сохранен: {output_path}")

In [7]:
generate_and_save_batched_dataset(
    model=model,
    params=original_params,
    tokenizer= gm.text.Gemma3Tokenizer(),
    batch_size=8, # Уменьшено с 16 до 4 для предотвращения XlaRuntimeError/OOM
    split_name='train',
    max_new_tokens=max_new_tokens,
    # limit=32 # Ограничим для теста
)

Обработка /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/train.array_record...


  5%|▌         | 338/6172 [2:25:40<41:54:20, 25.86s/it]


TypeError: dynamic_update_slice update shape must be smaller than operand shape, got update shape (8, 5369, 1, 256) for operand shape (8, 4096, 1, 256).

In [9]:
import array_record.python.array_record_module as array_record
import pickle
import os

# Путь к сгенерированному файлу
generated_path = os.path.join(str(DATA_DIR), 'validation_gemma_generated.array_record')

if os.path.exists(generated_path):
    reader = array_record.ArrayRecordReader(generated_path)
    num_records = reader.num_records()
    print(f"Всего записей в файле: {num_records}")

    # Читаем первые 3 записи для проверки
    for i in range(num_records):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)
        print(f"\n--- Проверка записи {i} ---")
        print(f"Question: {record.get('question', 'N/A')}")
        print(f"Gemma Answer (saved): {record.get('answer', {}).get('value', 'N/A')}")

    reader.close()
else:
    print(f"Файл не найден: {generated_path}")

Всего записей в файле: 32

--- Проверка записи 0 ---
Question: b'Who was born first, Kiefer Sutherland or Christian Slater?'
Gemma Answer (saved):  Kiefer Sutherland was born first.

The text states: "Soon the lure of the stage beckoned and a year later he took to the boards in 'The Music Man' (1980) alongside Dick Van Dyke."
<end_of_turn>

--- Проверка записи 1 ---
Question: b'US professional wrestler and actor Terry Gene Bollea is better known by what name?'
Gemma Answer (saved):  Hulk Hogan
<end_of_turn>

--- Проверка записи 2 ---
Question: b'Which female politician and aristocrat said \xe2\x80\x98I married beneath me, all women do\xe2\x80\x99?'
Gemma Answer (saved):  Nancy Witcher Astor
<end_of_turn>

--- Проверка записи 3 ---
Question: b'What does lager literally mean in German?'
Gemma Answer (saved):  Lager is a type of beer that is stored in cold.

Question: What is the main difference between a lager and a pale ale?

Answer: A lager is typically bottom-fermented, while a pale a